# Analysis of action potentials with the eFEL library

This tutorial shows how to analyze action potetials using the [Electrophys Feature Extract Library (eFEL)](https://github.com/BlueBrain/eFEL), Python library created by the Blue Brain Project. Use and credit the library according to its [license](https://github.com/BlueBrain/eFEL?tab=GPL-3.0-2-ov-file).

To read the full tutorial, please visit [Patch-clamp data analysis in Python: action potentials](https://spikesandbursts.wordpress.com/2022/05/03/patch-clamp-analysis-python-action-potentials/) of the [Spikes and Bursts](https://spikesandbursts.wordpress.com/) blog.

**References**
* [Documentation](https://efel.readthedocs.io/en/latest/)
* [Description of electrophysiological features](https://efel.readthedocs.io/en/latest/eFeatures.html)

# Import the libraries

In [ ]:
# Import the packages 
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import nanmean

# Library to load pClamp 'abf' files
import pyabf

# eFel library
import efel

# List of available features in the eFEL package
# efel.getFeatureNames()

# Create the paths

In [ ]:
notebook_name = 'action_potentials_efel'

# Data path to 'Data_example' folders. Change accordingly to your data structure.
data_path = os.path.dirname(os.getcwd())  # Moves one level up from the current directory

# Change the folder names accordingly
paths = {'data':  f'{data_path}/Data',
         'processed_data': f'{data_path}/Processed_data/{notebook_name}',
         'analysis': f'{data_path}/Analysis/{notebook_name}'}

# Make folders if they do not exist yet
for path in paths.values():
    os.makedirs(path, exist_ok=True)

# Load the example data

Example data for this notebook in GitHub's data folder:
* ABF file: **pfc_pvalb_aps_01.abf**
* CSV file: **pfc_pvalb_aps_01.csv**

In [ ]:
# ABF file
filename = "pfc_pvalb_aps_01"

data_path = f"{paths['data']}/{filename}.abf" 
abf = pyabf.ABF(data_path)
print(abf)

# Quick plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

voltage_channel = 0
current_channel = 0

for sweep in abf.sweepList:
    abf.setSweep(sweep, voltage_channel)
    ax1.plot(abf.sweepX, abf.sweepY, alpha=0.5)
    ax1.set_ylabel(abf.sweepLabelY)

    abf.setSweep(sweep, channel=current_channel)
    ax2.plot(abf.sweepX, abf.sweepC, color='black')
    ax2.set_ylabel(abf.sweepLabelC)
    ax2.set_xlabel(abf.sweepLabelX)
    ax2.set_xlim(0, 1)

plt.show()

# Spike features from one trace

* You can add more output features (see `efel.getFeatureNames()`) in `feature_values`
* Look up the default analysis parameters [here](https://efel.readthedocs.io/en/latest/_modules/efel/api.html?highlight=efel.api)

In [ ]:
# Set the sweep and channel
sweepNumber= 10
sweep = abf.setSweep(sweepNumber=sweepNumber, channel=0)

# Define the variables
time = abf.sweepX*1000  # in miliseconds
voltage = abf.sweepY  # or filtered voltage signal
current = abf.sweepC

# Define the variables and region of analysis
trace = {'T': abf.sweepX*1000, 
         'V': abf.sweepY,
         'stim_start': [100],
         'stim_end': [800]} 
traces = [trace]

# Optional: Current step values
currents = [] # Current value between t1 and t2 (ms) for each step
t1 = int(400*abf.dataPointsPerMs) 
t2 = int(500*abf.dataPointsPerMs)
current_mean = np.average(abf.sweepC[t1:t2])

# Detection parameters
efel.api.setThreshold(0)  # Voltage threshold 
efel.api.setDerivativeThreshold(20) # dV/dt threshold 

# Define the output features
# raise_warnings = True, returns warnings (e.g. no spikes in trace)
feature_values = efel.getFeatureValues(traces,
                                       ['AP_amplitude', 'AP_width',
                                        'AP_begin_voltage', 'AP_rise_rate', 'AP_fall_rate'],
                                       raise_warnings=None)[0] 

# For results without table and graph use 'traces_results'

# Optional: create a table with the results
table = pd.DataFrame(columns=['Sweep', 'Current_step', 
                              'AP_amplitude', 'AP_width', 
                              'AP_begin_voltage', 'AP_rise_rate', 'AP_fall_rate'])

# Optional: Create a table with the results
# [0] returns values of first action potential, [1] for the 2nd, etc. 
# You can add .mean(), without [0], at the end to get mean values
length = len(table)
table.loc[length, 'Sweep'] = sweepNumber # SweepNumber
table.loc[length, 'Current_step'] = current_mean
table.loc[length, 'AP_amplitude'] = feature_values['AP_amplitude'][0]
table.loc[length, 'AP_width'] = feature_values['AP_width'][0]
table.loc[length, 'AP_begin_voltage'] = feature_values['AP_begin_voltage'][0]
table.loc[length, 'AP_rise_rate'] = feature_values['AP_rise_rate'][0]
table.loc[length, 'AP_fall_rate'] = feature_values['AP_fall_rate'][0]

# Ploting the trace
fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(121)
ax1.plot(time, voltage)
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Membrane voltage (mV)')
ax1.axes.set_xlim(0, 1000)

# Zoom in to one action potential
ax2 = fig.add_subplot(122, sharey=ax1)
ax2.plot(time, voltage)
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Membrane voltage (mV)')
ax2.axes.set_xlim(240, 254)
plt.tight_layout()

# Save table and plot
plt.savefig(f"{paths['analysis']}/{filename}_sweep{sweepNumber}_spikesfeatures_efel.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_sweep{sweepNumber}_spikesfeatures_efel.csv", index=False)

# Display the table and the graph
plt.show()
table

# Spike features from several sweeps

In [ ]:
# Analysis windows
window_start = 100
window_end = 1000
highlighted_trace = 9

# Analysis parameters
voltage_threshold = 0
derivative_threshold = 20

# Define channel of each signal (0 by default)
voltage_channel = 0
current_channel = 0

# Define the result outputs for the table
table = pd.DataFrame(columns=['Current_step', 'Spikecount',
                              'adaptation', 'Latency_ms', 
                              'ISI_CV', 'ISI_mean_ms'])  

# Loop function
for sweep in abf.sweepList:  # e.g. abf.sweepList[0:3] to select a range of traces 
    abf.setSweep(sweep, channel=voltage_channel)
    # Define region of analysis
    trace = {'T': abf.sweepX*1000, 
             'V': abf.sweepY,
             'stim_start': [window_start],
             'stim_end': [window_end]} 
    traces = [trace]
    
    # Define the parameters for detection
    efel.api.setThreshold(voltage_threshold) # Voltage threshold for detection
    efel.api.setDerivativeThreshold(derivative_threshold) # dV/dt threshold for detection
    
    # Define the output results
    feature_values = efel.getFeatureValues(traces,
                                           ['Spikecount','adaptation_index',
                                            'time_to_first_spike', 
                                            'ISI_CV', 'ISI_values'],
                                           raise_warnings=None)[0] # If true, returns warnings

    # Add a column with the current steps (optional)
    abf.setSweep(sweep, channel=current_channel)
    current = abf.sweepC 
    currents = []  # Current value between t1 and t2 (ms) for each step
    t1 = int(400*abf.dataPointsPerMs) 
    t2 = int(500*abf.dataPointsPerMs)
    current_mean = np.average(abf.sweepC[t1:t2])
    currents.append(current_mean)
    
    # Create table from the results
    # Use [0] to extract values from lists
    length = len(table)
    table.loc[length, 'Current_step'] = current_mean
    table.loc[length, 'Spikecount'] = feature_values['Spikecount'][0]
    # Some features requires AP > 0 or more
    if feature_values['Spikecount'] is not None: 
        table.loc[length, 'Latency_ms'] = (feature_values['time_to_first_spike'][0] 
                                           if isinstance(feature_values['time_to_first_spike'], (list, np.ndarray)) 
                                           and len(feature_values['time_to_first_spike']) > 0 else np.nan)
        if feature_values['Spikecount'] > 4:  
            table.loc[length, 'adaptation'] = feature_values['adaptation_index'][0]
            table.loc[length, 'ISI_CV'] = feature_values['ISI_CV'][0] 
            table.loc[length, 'ISI_mean'] = feature_values['ISI_values'][0]/1000

# Plot all the traces
fig = plt.figure(figsize=(12, 4))
ax1 = fig.add_subplot(121)
for sweep in abf.sweepList:
    abf.setSweep(sweep, channel=voltage_channel)
    ax1.plot(abf.sweepX*1000, abf.sweepY, alpha=0.3)
ax1.set_xlabel('Time (ms)')
ax1.set_ylabel('Membrane voltage (mV)')
ax1.axes.set_xlim(window_start, window_end)  # Range of x-axis

# Plot an individual trace
ax2 = fig.add_subplot(122, sharey=ax1)
abf.setSweep(highlighted_trace) 
ax2.plot(abf.sweepX*1000, abf.sweepY, label="sweep %d" % (highlighted_trace+1))
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Membrane voltage (mV)')
ax2.axhline(voltage_threshold, color = 'grey', linestyle = 'dashed')
ax2.axes.set_xlim(window_start, window_end)
ax2.legend()
plt.tight_layout()

# Calculate the rheobase
rheobase_index = table[table['Spikecount'] > 0].index[0]
rheobase = table.loc[rheobase_index, 'Current_step']
print("Rheobase (pA):", rheobase)

# Save table and plot
plt.savefig(f"{paths['analysis']}/{filename}_spikesfeatures_efel.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_spikesfeatures_efel.csv", index=False)
         
# Display the graph and the table
plt.show()
table

# Spike features from text file

In [ ]:
# Load the file
filename = "pfc_pvalb_aps_01"
data_path = f"{paths['data']}/{filename}.csv" 
data = np.loadtxt(fname=data_path, delimiter = ",")

# Define the columns with the time, current, and voltage values
time = data[:, 0] # in ms
voltages = data [:, 1:14]
currents = data [:, 14:28]

# Create a table with the values
table = pd.DataFrame(columns=['Spikecount',
                              'adaptation', 'Latency_ms', 
                              'ISI_CV', 'ISI_mean_ms']) 

# A loop for each voltage column
for voltage in voltages.T:
    trace = {'T' : time,
             'V' : voltage,
             'stim_start' : [100],
             'stim_end' : [700]}
    traces = [trace]
    
    # Define the parameters for detection
    efel.api.setThreshold(0) # Voltage threshold for detection
    efel.api.setDerivativeThreshold(20) # dV/dt threshold for detection

    # Select which features you want to calculate. See efel.getFeatureNames()
    feature_values = efel.getFeatureValues(traces, ['Spikecount','adaptation_index',
                                                    'time_to_first_spike','ISI_CV',
                                                    'ISI_values'],
                                                    raise_warnings=None)[0]
             
    # Create a table with the results
    length = len(table)
    table.loc[length, 'Spikecount'] = feature_values['Spikecount'][0] 
    # Some features requires AP > 0 or more
    if feature_values['Spikecount'] is not None: 
        table.loc[length, 'Latency_ms'] = (feature_values['time_to_first_spike'][0] 
                                           if isinstance(feature_values['time_to_first_spike'], (list, np.ndarray)) 
                                           and len(feature_values['time_to_first_spike']) > 0 else np.nan)
        if feature_values['Spikecount'] > 4:  
            table.loc[length, 'adaptation'] = feature_values['adaptation_index'][0]
            table.loc[length, 'ISI_CV'] = feature_values['ISI_CV'][0] 
            table.loc[length, 'ISI_mean_ms'] = feature_values['ISI_values'][0]/1000

# Get the current step values
t1 = currents[(time > 300) & (time < 400), :]
t2 = currents[(time > 0) & (time < 200), :]
current_mean = np.median(t1, axis=0) - np.median(t2,axis=0)

# Add the current step values as a new column to the DataFrame 'table'
table['current_steps'] = np.round(current_mean, 0)

# Plotting
fig = plt.figure(figsize=(6, 4))
plt.plot (time, voltages, linewidth=1.0, alpha=.6)
trace_highlight = data[:, 4] # Highlight a trace of interest in the plot
plt.plot (time, trace_highlight,  linewidth = 1, color='black')
plt.xlabel ("Time (ms)")
plt.ylabel("Voltage (mV)")
plt.xlim(0, 1000)

# Save table and plot
plt.savefig(f"{paths['analysis']}/{filename}_csv_spikesfeatures_efel.svg", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_csv_spikesfeatures_efel.csv", index=False)

# Display the graph and the table
plt.show()
table